# A. Kaggle CPU-only platform baseline

**Accelerator: None** · fresh Kaggle session · Python 3.13 · generated by `scripts/build_kaggle_diagnostic_notebooks.py`

No Stokastisk, no CUDA, PyTorch, NVML or vLLM import, no Enigma connector, no runtime token, no Git clone. Question: does a plain CPU session survive, and does ordinary HTTPS work, with NONE of our software present?

**Evidence durability.** Every step prints flushed `MARK` and `VERDICT` lines and appends them to an fsynced `markers.jsonl` in a new, never-reused attempt directory. After a complete platform termination the VM's files may be gone (on 2026-11-26 a new session had to re-clone everything), but cell output printed before the failure was kept in the saved notebook. **Missing files do not prove that no crash occurred.** After a fatal session, copy the last `MARK` lines (UTC time, `t` = seconds since the prelude) into the incident report.

Run the cells top to bottom (or *Run all*). Every step ends with `VERDICT <step>: PASS | FAIL | INCONCLUSIVE`; anything but PASS raises and stops the run. Then run the final **evidence export** cell manually.

In [ ]:
# P0. Prelude: environment guard, attempt id, markers and verdicts. Standard library only: no network, no GPU library is imported.
import os, sys, json, time, uuid, platform, shutil, subprocess, re
NOTEBOOK = "enigma_kaggle_cpu_platform_baseline"
EXPECT = "cpu"                                   # "cpu" = Accelerator None; "gpu" = GPU T4 x2
assert os.path.isdir("/kaggle/working"), "Kaggle only: refusing to run anywhere else (never on a developer laptop)"


class DiagnosticStop(Exception):
    """Raised by every non-PASS verdict so that 'Run all' stops here. Later cells must never run after a failure."""


HAS_GPU_DEVICE = os.path.exists("/dev/nvidia0")
if EXPECT == "cpu" and HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. A GPU is attached; set Accelerator = None and start a NEW session")
if EXPECT == "gpu" and not HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. No GPU; set Accelerator = GPU T4 x2 and start a NEW session")
assert sys.version_info[:2] == (3, 13), f"expected Python 3.13 (build-5 wheel is cp313), got {platform.python_version()}"

ATTEMPT = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "-" + uuid.uuid4().hex[:8]
ATTEMPT_DIR = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}/{ATTEMPT}"
os.makedirs(ATTEMPT_DIR, exist_ok=False)               # a new directory per attempt: earlier evidence is never overwritten
T0 = time.monotonic()
RESULTS = {}                                           # verdicts of THIS kernel session only (stale outputs can never satisfy a check)


def _read(path):
    try:
        with open(path) as fh:
            return fh.read(65536)
    except OSError:
        return None


def host():
    """Host memory, cgroup v2 memory (current, max, events), shared memory, disk, uptime and load. No environment variables are read."""
    h = {}
    for line in (_read("/proc/meminfo") or "").splitlines():
        k, _, v = line.partition(":")
        if k in ("MemTotal", "MemAvailable", "Shmem"):
            h[k.lower() + "_mib"] = round(int(v.split()[0]) / 1024, 1)
    mib = lambda s: None if s is None or not s.strip().isdigit() else round(int(s) / 2**20, 1)   # noqa: E731 ("max" = unlimited -> None)
    h["cgroup_current_mib"], h["cgroup_max_mib"] = mib(_read("/sys/fs/cgroup/memory.current")), mib(_read("/sys/fs/cgroup/memory.max"))
    h["cgroup_events"] = {k: int(v) for k, _, v in (l.partition(" ") for l in (_read("/sys/fs/cgroup/memory.events") or "").splitlines()) if v.strip().isdigit()}
    for d in ("/kaggle/working", "/tmp", "/dev/shm"):
        try:
            u = shutil.disk_usage(d)
            h["disk_" + d.strip("/").replace("/", "_")] = {"used_gib": round(u.used / 2**30, 2), "free_gib": round(u.free / 2**30, 2)}
        except OSError:
            pass
    up = _read("/proc/uptime")
    h["vm_uptime_s"] = float(up.split()[0]) if up else None
    h["loadavg"] = os.getloadavg()
    return h


def ident():
    return {"pid": os.getpid(), "ppid": os.getppid(), "pgid": os.getpgid(0), "sid": os.getsid(0)}


def mark(name, **fields):
    rec = {"attempt": ATTEMPT, "nb": NOTEBOOK, "marker": name, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "t": round(time.monotonic() - T0, 2), **ident(), **host(), **fields}
    line = json.dumps(rec, default=str)
    with open(f"{ATTEMPT_DIR}/markers.jsonl", "a") as fh:
        fh.write(line + "\n"); fh.flush(); os.fsync(fh.fileno())
    print("MARK", line, flush=True)
    return rec


def verdict(step, status, reason, **evidence):
    assert status in ("PASS", "FAIL", "INCONCLUSIVE"), status
    RESULTS[step] = {"status": status, "reason": reason, "attempt": ATTEMPT, **evidence}
    mark("verdict", step=step, status=status, reason=reason, **evidence)
    print(f"VERDICT {step}: {status} | {reason}", flush=True)
    if status != "PASS":
        raise DiagnosticStop(f"{step} {status}: {reason}")


def require(*steps):
    """Every prerequisite must have PASSED in this very session (same attempt id)."""
    bad = [s for s in steps if RESULTS.get(s, {}).get("status") != "PASS" or RESULTS[s].get("attempt") != ATTEMPT]
    if bad:
        raise DiagnosticStop(f"INCONCLUSIVE: prerequisite(s) did not PASS in this session: {bad}. Start a new session and run from the top.")


def observe(step, seconds, every=15.0, sample=None):
    """Survival observation: one MARK every `every` s. PASS only if the full duration elapsed with no cgroup OOM kill and no failed sample check."""
    start, oom0, n, problems = time.monotonic(), host()["cgroup_events"].get("oom_kill"), 0, []
    mark(step + "_begin", seconds=seconds, every=every)
    while time.monotonic() - start < seconds:
        time.sleep(min(every, max(0.0, seconds - (time.monotonic() - start))))
        n += 1
        extra = sample() if sample else {}
        if extra.get("problem"):
            problems.append(extra["problem"])
        mark(step + "_alive", i=n, **extra)
    el, oom1 = time.monotonic() - start, host()["cgroup_events"].get("oom_kill")
    delta = None if oom0 is None or oom1 is None else oom1 - oom0
    ok = el >= seconds - 1.0 and delta in (None, 0) and not problems
    verdict(step, "PASS" if ok else "FAIL", f"observed {el:.0f}/{seconds}s, oom_kill delta {delta}, problems {problems[:3]}", observed_s=round(el, 1),
            oom_kill_delta=delta, samples=n)


def export_evidence():
    """Zip this attempt's directory (only it). Safe to run after a failure."""
    base = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}-{ATTEMPT}"
    path = shutil.make_archive(base, "zip", ATTEMPT_DIR)
    print("evidence archive:", path, os.path.getsize(path), "bytes", flush=True)
    return path


mark("prelude", python=platform.python_version(), expect=EXPECT, gpu_device=HAS_GPU_DEVICE)
print("ATTEMPT", ATTEMPT, "| markers:", ATTEMPT_DIR + "/markers.jsonl", flush=True)

## A1. Static inventory (no network)

In [ ]:
import importlib.metadata as md
mark("A1_inventory", cpus=os.cpu_count(), packages=sum(1 for _ in md.distributions()), kernel=platform.release())
verdict("A1_inventory", "PASS", "recorded")

## A2. Idle survival control: 10 minutes, nothing running
If the session dies here, the cause is Kaggle-side: none of Enigma, Stokastisk, vLLM or CUDA is present. **Do not continue to the network tests in that case.**

In [ ]:
require("A1_inventory")
observe("A2_idle_10min", 600, every=15)

## A3. Unauthenticated HTTPS: public controls and the Enigma health endpoint
6 rounds, 10 s apart, 20 s timeout, at most 64 KiB read per response. A failing Enigma request while the controls succeed is an Enigma-side network FAIL; failing controls too = INCONCLUSIVE (Kaggle internet). A session that dies here leaves no verdict: its last MARK says which request was last completed.

In [ ]:
require("A2_idle_10min")
import urllib.request, urllib.error
TARGETS = {"control_pypi": "https://pypi.org/simple/pip/", "control_gstatic": "https://www.gstatic.com/generate_204",
           "enigma_health": "https://enigma-api-production-c8cffe9d.up.railway.app/api/v1/health"}
rows = []
for rnd in range(6):
    for name, url in TARGETS.items():
        t = time.monotonic()
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Enigma-Kaggle-Diagnostic/2.0", "Accept": "*/*"})
            with urllib.request.urlopen(req, timeout=20) as resp:
                n_bytes = len(resp.read(65536))
                rows.append({"round": rnd + 1, "target": name, "http": resp.status, "bytes": n_bytes, "s": round(time.monotonic() - t, 3)})
        except urllib.error.HTTPError as e:
            rows.append({"round": rnd + 1, "target": name, "http": e.code, "s": round(time.monotonic() - t, 3)})
        except Exception as e:
            rows.append({"round": rnd + 1, "target": name, "error": type(e).__name__, "s": round(time.monotonic() - t, 3)})
        mark("A3_request", **rows[-1])
    if rnd < 5:
        mark("A3_sleep_begin", round=rnd + 1)
        time.sleep(10)
ok = lambda r: r.get("http") in (200, 204)                                    # noqa: E731
controls = [r for r in rows if r["target"].startswith("control")]
enigma = [r for r in rows if r["target"] == "enigma_health"]
if not all(map(ok, controls)):
    verdict("A3_https", "INCONCLUSIVE", f"public controls failed {sum(not ok(r) for r in controls)}/{len(controls)}: check that Internet is ON")
verdict("A3_https", "PASS" if all(map(ok, enigma)) else "FAIL", f"enigma {sum(map(ok, enigma))}/{len(enigma)} ok, controls all ok",
        enigma_ok=sum(map(ok, enigma)), enigma_total=len(enigma))

## A4. Post-network idle survival: 5 minutes
The earlier fatal sessions ended seconds after network activity; this checks the same window with no other software.

In [ ]:
require("A3_https")
observe("A4_post_network_idle_5min", 300, every=15)

## A5. Optional, after A1–A4 PASS: dependency plan on the CPU image (dry run, installs nothing)
Answers whether the build-5 install would change the same packages on a CPU image. `pip install --dry-run` resolves only; nothing is installed.

In [ ]:
require("A4_post_network_idle_5min")
report = f"{ATTEMPT_DIR}/pip-dry-run-report.json"
spec = "stokastisk @ https://huggingface.co/waqasm86/stokastisk/resolve/main/wheels/stokastisk-0.1.0-5-cp313-cp313-linux_x86_64.whl#sha256=fe80d7c36fa78cf8a26feb1e00bb5f1f006ca45a027450353603976320ae0b3e"
r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "--quiet", "--report", report, "--index-url", "https://pypi.org/simple", spec],
                   capture_output=True, text=True)
if r.returncode != 0:
    mark("A5_dry_run", rc=r.returncode, stderr_tail=r.stderr[-1500:])
    verdict("A5_dry_run", "INCONCLUSIVE", "the CPU image cannot resolve the build-5 requirements (e.g. torch==2.11.0+cu130 not installed/available): see stderr_tail")
import importlib.metadata as md
have = {re.sub(r"[-_.]+", "-", d.metadata["Name"]).lower(): d.version for d in md.distributions() if d.metadata["Name"]}
plan = {re.sub(r"[-_.]+", "-", i["metadata"]["name"]).lower(): i["metadata"]["version"] for i in json.load(open(report))["install"]}
replaced = {k: [have[k], v] for k, v in plan.items() if k in have and have[k] != v}
mark("A5_dry_run", would_install=len(plan), replaced=replaced, added=sorted(k for k in plan if k not in have))
verdict("A5_dry_run", "PASS", f"{len(plan)} would be installed; would replace {replaced}", replaced=replaced)

## Evidence export

In [ ]:
# Evidence export (run it even after a FAIL / INCONCLUSIVE stop). Review the archive before sharing it; it contains no credential.
mark("export_begin", results={k: v["status"] for k, v in RESULTS.items()})
archive = export_evidence()
print(json.dumps({k: {"status": v["status"], "reason": v["reason"]} for k, v in RESULTS.items()}, indent=1), flush=True)